In [2]:
# ==========================================================
# STEP 1: SETUP & CREATE MESSY RAW DATASET
# ==========================================================

import pandas as pd
import numpy as np

# Messy raw export from a university LMS portal
raw_lms_data = {
    'student_id': [101, 102, 103, 104, 105, 105, 106, 107, 108, 109],
    'student_name': [
        '  Amina Okafor ',    # Leading and trailing spaces
        'JAMES SMITH',         # ALL CAPS
        'fatima bello',        # all lowercase
        'David Chen',          # Normal
        'Grace Eze',           # Duplicate row 1
        'Grace Eze',           # Duplicate row 2 (identical duplicate!)
        'Omar Hassan ',        # Trailing space
        'Sarah Williams',      # Normal
        np.nan,                # MISSING NAME!
        'Kwame Asante'         # Normal
    ],
    'email': [
        'AMINA@SCHOOL.EDU',    # Uppercase email
        'james.smith@school.edu',
        'fatima@school.edu',
        'david.chen@gmail.com',# External email domain
        'grace@school.edu',
        'grace@school.edu',
        'omar@school.edu',
        'sarah@school.edu',
        'unknown@school.edu',
        'kwame@school.edu'
    ],
    'course_code': [
        'CS101', 'cs101', 'CS101 ', 'MATH201', 'CS101', 
        'CS101', 'MATH201', 'cs101', 'MATH201', 'CS101'
    ],
    'raw_score': [
        '88.5%',   # String with % sign
        '92.0%',   # String with % sign
        np.nan,    # MISSING SCORE! (Student was absent)
        '78.0%', 
        '55.0%', 
        '55.0%', 
        '42.0%', 
        '96.0%', 
        '80.0%', 
        'INVALID'  # Corrupted data entry!
    ]
}

df_raw = pd.DataFrame(raw_lms_data)

print("⚠️ RAW UNCLEANED LMS EXPORT (Look at the inconsistencies!):")
df_raw

⚠️ RAW UNCLEANED LMS EXPORT (Look at the inconsistencies!):


,student_id,student_name,email,course_code,raw_score
0,101,Amina Okafor,AMINA@SCHOOL.EDU,CS101,88.5%
1,102,JAMES SMITH,james.smith@school.edu,cs101,92.0%
2,103,fatima bello,fatima@school.edu,CS101,NaN
3,104,David Chen,david.chen@gmail.com,MATH201,78.0%
4,105,Grace Eze,grace@school.edu,CS101,55.0%
5,105,Grace Eze,grace@school.edu,CS101,55.0%
6,106,Omar Hassan,omar@school.edu,MATH201,42.0%
7,107,Sarah Williams,sarah@school.edu,cs101,96.0%
8,108,NaN,unknown@school.edu,MATH201,80.0%
9,109,Kwame Asante,kwame@school.edu,CS101,INVALID


In [3]:
# ==========================================================
# STEP 2: DIAGNOSING DATA ISSUES
# ==========================================================

print("--- 1. DATA TYPES & MISSING VALUE AUDIT ---")
# .info() reveals which columns have missing values and their data types
df_raw.info()

print("\n--- 2. COUNT OF MISSING VALUES PER COLUMN ---")
# .isna().sum() counts how many empty (NaN) cells exist in each column
print(df_raw.isna().sum())

print("\n--- 3. DUPLICATE ROWS CHECK ---")
# .duplicated().sum() counts how many exact duplicate rows exist
print(f"Total duplicate rows found: {df_raw.duplicated().sum()}")

print("\n--- 4. UNIQUE COURSE CODES (Notice the duplicates due to casing/spaces!) ---")
# .unique() shows every distinct value found in the column
print(df_raw['course_code'].unique())

--- 1. DATA TYPES & MISSING VALUE AUDIT ---
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   student_id    10 non-null     int64
 1   student_name  9 non-null      str  
 2   email         10 non-null     str  
 3   course_code   10 non-null     str  
 4   raw_score     9 non-null      str  
dtypes: int64(1), str(4)
memory usage: 916.0 bytes

--- 2. COUNT OF MISSING VALUES PER COLUMN ---
student_id      0
student_name    1
email           0
course_code     0
raw_score       1
dtype: int64

--- 3. DUPLICATE ROWS CHECK ---
Total duplicate rows found: 1

--- 4. UNIQUE COURSE CODES (Notice the duplicates due to casing/spaces!) ---
<ArrowStringArray>
['CS101', 'cs101', 'CS101 ', 'MATH201']
Length: 4, dtype: str


In [4]:
# ==========================================================
# STEP 3: STRING STANDARDIZATION
# ==========================================================

# Make a working copy of our raw data
df_clean = df_raw.copy()
# 1. Strip spaces and fix Name casing (e.g., '  Amina Okafor ' -> 'Amina Okafor')
df_clean['student_name'] = df_clean['student_name'].str.strip().str.title()
# 2. Standardize emails to lowercase
df_clean['email'] = df_clean['email'].str.strip().str.lower()
# 3. Standardize course codes: remove spaces and make ALL UPPERCASE ('cs101 ' -> 'CS101')
df_clean['course_code'] = df_clean['course_code'].str.strip().str.upper()
print("--- AFTER TEXT CLEANING ---")
print("Unique course codes now (Clean & Unified!):", df_clean['course_code'].unique())
df_clean[['student_name', 'email', 'course_code']]

--- AFTER TEXT CLEANING ---
Unique course codes now (Clean & Unified!): <ArrowStringArray>
['CS101', 'MATH201']
Length: 2, dtype: str


,student_name,email,course_code
0,Amina Okafor,amina@school.edu,CS101
1,James Smith,james.smith@school.edu,CS101
2,Fatima Bello,fatima@school.edu,CS101
3,David Chen,david.chen@gmail.com,MATH201
4,Grace Eze,grace@school.edu,CS101
5,Grace Eze,grace@school.edu,CS101
6,Omar Hassan,omar@school.edu,MATH201
7,Sarah Williams,sarah@school.edu,CS101
8,NaN,unknown@school.edu,MATH201
9,Kwame Asante,kwame@school.edu,CS101


In [6]:
# ==========================================================
# STEP 4: CONVERTING CORRUPTED STRINGS TO NUMBERS
# ==========================================================

# 1. Strip off the '%' sign from the text
clean_score_text = df_clean['raw_score'].str.replace('%', '', regex=False)
# 2. Convert to float numbers; turn 'INVALID' into NaN automatically
df_clean['final_score'] = pd.to_numeric(clean_score_text, errors='coerce')
print("--- BEFORE vs AFTER SCORE CONVERSION ---")
print(pd.DataFrame({
    'Original_Raw': df_raw['raw_score'],
    'Clean_Float': df_clean['final_score']
}))
print("\nData type of clean 'final_score':", df_clean['final_score'].dtype)

--- BEFORE vs AFTER SCORE CONVERSION ---
  Original_Raw  Clean_Float
0        88.5%         88.5
1        92.0%         92.0
2          NaN          NaN
3        78.0%         78.0
4        55.0%         55.0
5        55.0%         55.0
6        42.0%         42.0
7        96.0%         96.0
8        80.0%         80.0
9      INVALID          NaN

Data type of clean 'final_score': float64


In [7]:
# ==========================================================
# STEP 5: DEDUPLICATION & MISSING VALUE RESOLUTION
# ==========================================================

# 1. REMOVE DUPLICATE ROWS
print(f"Rows Before Dropping Duplicates: {len(df_clean)}")
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows After Dropping Duplcates: {len(df_clean)}")

# 2. FILL MISSING NAMES
df_clean['student_name'] = df_clean['student_name'].fillna('Unknown Student')

# 3. FILL MISSING SCORES WITH CLASS AVERAGE
class_average = df_clean['final_score'].mean()
print(f"\nCalculated Class Average Score: {class_average:.1f}%")

# .fillna() replaces NaN with our calculated average
df_clean['final_score'] = df_clean['final_score'].fillna(class_average).round(1)

# Drop the old uncleaned raw_score column
df_clean = df_clean.drop(columns=['raw_score'])

print("\n✅ FULLY CLEANED & VALIDATED DATASET:")
df_clean

Rows Before Dropping Duplicates: 10
Rows After Dropping Duplcates: 9

Calculated Class Average Score: 75.9%

✅ FULLY CLEANED & VALIDATED DATASET:


,student_id,student_name,email,course_code,final_score
0,101,Amina Okafor,amina@school.edu,CS101,88.5
1,102,James Smith,james.smith@school.edu,CS101,92.0
2,103,Fatima Bello,fatima@school.edu,CS101,75.9
3,104,David Chen,david.chen@gmail.com,MATH201,78.0
4,105,Grace Eze,grace@school.edu,CS101,55.0
5,106,Omar Hassan,omar@school.edu,MATH201,42.0
6,107,Sarah Williams,sarah@school.edu,CS101,96.0
7,108,Unknown Student,unknown@school.edu,MATH201,80.0
8,109,Kwame Asante,kwame@school.edu,CS101,75.9


In [8]:
# ==========================================================
# STEP 6: THE REUSABLE DATA CLEANING PIPELINE
# ==========================================================

def clean_lms_export(df_dirty):
    """
    Automated Data Sanitization Pipeline:
    - Removes whitespace and standardizes casing
    - Strips special characters from numeric columns
    - Drops duplicate records
    - Safely imputes missing values
    """
    df = df_dirty.copy()
    
    # 1. Deduplicate
    df = df.drop_duplicates().reset_index(drop=True)
    
    # 2. Clean Text
    if 'student_name' in df.columns:
        df['student_name'] = df['student_name'].str.strip().str.title().fillna('Unknown Student')
        
    if 'email' in df.columns:
        df['email'] = df['email'].str.strip().str.lower()
        
    if 'course_code' in df.columns:
        df['course_code'] = df['course_code'].str.strip().str.upper()
        
    # 3. Clean Numeric Scores
    if 'raw_score' in df.columns:
        clean_scores = df['raw_score'].astype(str).str.replace('%', '', regex=False)
        df['final_score'] = pd.to_numeric(clean_scores, errors='coerce')
        avg_score = df['final_score'].mean()
        df['final_score'] = df['final_score'].fillna(avg_score).round(1)
        df = df.drop(columns=['raw_score'])
        
    return df

# TEST RUN: Clean the entire raw dataset in one command!
final_production_data = clean_lms_export(df_raw)
print("🚀 PRODUCTION-READY DATASET:")
final_production_data

🚀 PRODUCTION-READY DATASET:


,student_id,student_name,email,course_code,final_score
0,101,Amina Okafor,amina@school.edu,CS101,88.5
1,102,James Smith,james.smith@school.edu,CS101,92.0
2,103,Fatima Bello,fatima@school.edu,CS101,75.9
3,104,David Chen,david.chen@gmail.com,MATH201,78.0
4,105,Grace Eze,grace@school.edu,CS101,55.0
5,106,Omar Hassan,omar@school.edu,MATH201,42.0
6,107,Sarah Williams,sarah@school.edu,CS101,96.0
7,108,Unknown Student,unknown@school.edu,MATH201,80.0
8,109,Kwame Asante,kwame@school.edu,CS101,75.9


In [12]:
dirty_dept_data = pd.DataFrame({
    'teacher_id': [1, 2, 3, 4],
    'department': ['  computer science ', 'MATH ', 'math', 'COMPUTER SCIENCE']
})

dirty_dept_data['department'] = dirty_dept_data['department'].str.strip().str.upper()
print("Unique course codes now (Clean & Unified!):", dirty_dept_data['department'].unique())
dirty_dept_data

Unique course codes now (Clean & Unified!): <ArrowStringArray>
['COMPUTER SCIENCE', 'MATH']
Length: 2, dtype: str


,teacher_id,department
0,1,COMPUTER SCIENCE
1,2,MATH
2,3,MATH
3,4,COMPUTER SCIENCE
